# Python for ML4EO · Tutorial T2
# NumPy: images as arrays

**Self-study · about 75 minutes · after Tutorials T0 and T1**

A Sentinel-2 band is a grid of numbers: one reflectance value per pixel. The course image of Kigali has 1,702 rows × 2,380 columns, about 4 million pixels per band.
With the lists of T1 you would need a loop over 4 million pixels. **NumPy** computes on the whole grid at once, in one line.

| Part | Topic | You will use it for … |
|---|---|---|
| 1 | NumPy arrays | computing on many values at once |
| 2 | 2-D arrays: rows and columns | an image is a 2-D array |
| 3 | summary statistics | mean, minimum, maximum of a band |
| 4 | Boolean masks | selecting pixels with a condition (water, vegetation) |
| 5 | no-data values and NaN | pixels without data |
| 6 | a real Sentinel-2 image | NDVI, MNDWI and land-cover shares of a real image |
| 7 | stacking bands and building the feature matrix | the input table of a machine-learning model |
| 8 | common errors | shape mismatch, copies |

**Data:** the Sentinel-2 GeoTIFF files of 3 September 2026 in `ml4eo/data` (the same files as in the practicals). Part 6 reads them with a ready-made function; Tutorial T3 explains how that function works.
**Software:** Part 6 needs the library **rasterio** (see 6.2).

Tasks: **▶ RUN** · **✎ CHANGE** · **✍ EXERCISE** (followed by a check).

### Set-up  ▶ RUN

In [ ]:
# Run this cell once. It creates the function check(), which tells you whether an exercise is correct.
def check(correct, hint):
    if correct:
        print("✓ Correct, well done!")
    else:
        print("✗ Not yet. Hint:", hint)

### Import NumPy  ▶ RUN
By convention NumPy is imported with the short name `np` (T1, Part 9).

In [ ]:
import numpy as np

---
## Part 1 · NumPy arrays

### 1.1 From a list to an array  ▶ RUN
`np.array()` turns a list into an **array**.

In [ ]:
red = np.array([0.06, 0.10, 0.07, 0.15])
red

### 1.2 A list does not compute  ▶ RUN
Multiplying a **list** by 2 does not double the numbers; it repeats the list.

In [ ]:
red_list = [0.06, 0.10, 0.07, 0.15]
red_list * 2

### 1.3 An array computes element by element  ▶ RUN
The same operation on an **array** is applied to every element.

In [ ]:
red * 2

### 1.4 Two arrays together  ▶ RUN
With two arrays of the same length, `+ - * /` combine the elements **pair by pair**: the first with the first, the second with the second, …

In [ ]:
nir = np.array([0.42, 0.30, 0.05, 0.25])
nir - red

### 1.5 NDVI of four pixels in one line  ▶ RUN
The formula is written exactly as for one pixel (T1), but now computes four NDVI values at once. No loop is needed.

In [ ]:
ndvi = (nir - red) / (nir + red)
ndvi

### 1.6 Rounding an array  ▶ RUN
`np.round(array, digits)` rounds every element.

In [ ]:
np.round(ndvi, 2)

### 1.7 Your turn  ✍ EXERCISE
Green (B03) and short-wave infrared (B11) reflectance of four pixels are given.
Compute the MNDWI = (Green − SWIR) / (Green + SWIR) of all four pixels in **one line**.

In [ ]:
green = np.array([0.05, 0.08, 0.06, 0.09])
swir = np.array([0.02, 0.20, 0.25, 0.30])
mndwi = (green - swir) / (green + swir)       # ✍ replace None

check(mndwi is not None and np.allclose(mndwi, (green - swir) / (green + swir)), "mndwi = (green - swir) / (green + swir)")


Which of the four pixels is water? (MNDWI > 0.)

---
## Part 2 · 2-D arrays: rows and columns

### 2.1 A tiny image  ▶ RUN
An image is a **2-D array**: a list of rows, each row a list of values.
This "image" of the red band has 3 rows and 4 columns (12 pixels).

In [ ]:
tiny = np.array([[0.05, 0.06, 0.12, 0.15],
                 [0.04, 0.07, 0.11, 0.14],
                 [0.03, 0.05, 0.08, 0.20]])
tiny

### 2.2 Shape, dimensions and size  ▶ RUN
* `.shape` gives **(rows, columns)**. This is the first thing to check about any image.
* `.ndim` gives the number of dimensions; `.size` the number of pixels.

In [ ]:
print("shape:", tiny.shape)
print("dimensions:", tiny.ndim)
print("number of pixels:", tiny.size)

### 2.3 Data type  ▶ RUN
`.dtype` gives the kind of number stored: `float64` or `float32` (decimal numbers) or `int`/`uint` (whole numbers). Satellite bands are often stored as whole numbers, which matters in Part 5.

In [ ]:
tiny.dtype

### 2.4 One pixel: `[row, column]`  ▶ RUN
A pixel is selected with **two** indices: row first, column second. Counting starts at 0, as for lists (T1).

In [ ]:
print(tiny[0, 0])     # upper-left pixel
print(tiny[2, 3])     # row 2 (third row), column 3 (fourth column)
print(tiny[-1, -1])   # the last row, the last column: the same pixel

### 2.5 Rows and columns: slicing  ▶ RUN
A colon `:` alone means "all". So `tiny[0, :]` is the whole first row and `tiny[:, 1]` the whole second column.

In [ ]:
print(tiny[0, :])     # first row
print(tiny[:, 1])     # second column

### 2.6 Cutting out a window  ▶ RUN
`tiny[0:2, 1:3]` takes rows 0–1 and columns 1–2: a 2 × 2 window. As with lists, the stop number is **not** included.
This is how you cut a study area out of a large image.

In [ ]:
window = tiny[0:2, 1:3]
print(window)
print(window.shape)

### 2.7 Selecting pixels  ✍ EXERCISE
From `tiny`:
1. store the value of the pixel in the **second row, third column** in `pixel`;
2. store the **last row** (all columns) in `last_row`.

In [ ]:
pixel = tiny[1, 2]          # ✍ replace None
last_row = tiny[-1, :]       # ✍ replace None

check(pixel == 0.11, "second row = index 1, third column = index 2: tiny[1, 2]")
check(last_row is not None and np.array_equal(last_row, [0.03, 0.05, 0.08, 0.20]), "tiny[-1, :] or tiny[2, :]")


---
## Part 3 · Summary statistics

### 3.1 Statistics of the whole image  ▶ RUN
Arrays have built-in methods for statistics over **all** pixels.

In [ ]:
print("mean:   ", tiny.mean())
print("minimum:", tiny.min())
print("maximum:", tiny.max())
print("std:    ", tiny.std())     # standard deviation

### 3.2 Median and percentiles  ▶ RUN
The median and percentiles are NumPy **functions**: `np.median(array)`, `np.percentile(array, p)`.
The 2nd and 98th percentiles are used to stretch the contrast of satellite images (T5).

In [ ]:
print("median:", np.median(tiny))
print("98th percentile:", np.percentile(tiny, 98))

### 3.3 Statistics per row or per column: `axis`  ▶ RUN
`axis=0` computes **down** the rows (one value per column); `axis=1` computes **along** each row (one value per row).

In [ ]:
print("mean of each column:", tiny.mean(axis=0))
print("mean of each row:   ", tiny.mean(axis=1))

---
## Part 4 · Boolean masks

### 4.1 A condition on every pixel  ▶ RUN
A comparison (T1, Part 6) applied to an array gives an array of `True`/`False` of the same shape: a **mask**.
Here: which pixels of a small NDVI image are dense vegetation (NDVI > 0.5)?

In [ ]:
ndvi_img = np.array([[-0.20, 0.10, 0.62],
                     [ 0.55, 0.30, 0.71],
                     [-0.05, 0.48, 0.80]])
dense = ndvi_img > 0.5
dense

### 4.2 Counting with a mask  ▶ RUN
`True` counts as 1 and `False` as 0. So `.sum()` counts the pixels and `.mean()` gives the **share** (fraction) of pixels for which the condition is true.

In [ ]:
print("dense vegetation pixels:", dense.sum())
print("share of the image:", dense.mean())

### 4.3 Selecting values with a mask  ▶ RUN
`ndvi_img[dense]` keeps only the values where the mask is `True` (the result is 1-D: a list of values, no longer an image).

In [ ]:
ndvi_img[dense]

### 4.4 Combining conditions: `&` and `|`  ▶ RUN
For arrays, use `&` (and) and `|` (or) instead of `and`/`or`, and put **brackets around each condition**.

In [ ]:
sparse = (ndvi_img >= 0) & (ndvi_img <= 0.5)
sparse

If you write `ndvi_img >= 0 and ndvi_img <= 0.5` you get `ValueError: The truth value of an array … is ambiguous`. This error means: use `&` with brackets.

### 4.5 Classifying every pixel: `np.where`  ▶ RUN
`np.where(condition, value_if_true, value_if_false)` is the array version of `if … else`. Here: class 1 for water (NDVI < 0), class 0 otherwise.

In [ ]:
water_map = np.where(ndvi_img < 0, 1, 0)
water_map

### 4.6 Three classes  ▶ RUN
`np.where` can be nested: water = 0, sparse vegetation = 1, dense vegetation = 2.

In [ ]:
classes = np.where(ndvi_img < 0, 0, np.where(ndvi_img <= 0.5, 1, 2))
classes

### 4.7 Masks  ✍ EXERCISE
Using `ndvi_img`:
1. count the **water** pixels (NDVI < 0) and store the number in `n_water`;
2. compute the **mean NDVI of the dense vegetation pixels** (NDVI > 0.5) and store it in `mean_dense`.

In [ ]:
n_water = (ndvi_img < 0).sum()        # ✍ replace None
mean_dense = ndvi_img[ndvi_img > 0.5].mean()     # ✍ replace None

check(n_water == 2, "(ndvi_img < 0).sum()")
check(mean_dense is not None and abs(mean_dense - 0.67) < 1e-9, "select first, then average: ndvi_img[ndvi_img > 0.5].mean()")


---
## Part 5 · No-data values and NaN

### 5.1 Pixels without data
Not every pixel of an image has a measurement: the edge of the satellite swath, or pixels removed because of cloud. In the course images these pixels have the value **0**.
A reflectance of 0 is not a real measurement, so it must not enter a mean or an NDVI.

### 5.2 NaN: "not a number"  ▶ RUN
NumPy marks missing values with `np.nan`. Any calculation with NaN gives NaN.

In [ ]:
band = np.array([0.06, 0.0, 0.10, 0.0, 0.08])
band[band == 0] = np.nan      # replace every 0 by NaN
band

The line `band[band == 0] = np.nan` reads: "*in `band`, where `band` equals 0, put NaN*". It is exactly the line in `load_band()` of the practicals.

### 5.3 `mean` gives NaN  ▶ RUN

In [ ]:
band.mean()

### 5.4 `np.nanmean` ignores NaN  ▶ RUN
The functions `np.nanmean`, `np.nanmin`, `np.nanmax`, `np.nanmedian`, `np.nanpercentile` skip the NaN values.

In [ ]:
np.nanmean(band)

### 5.5 Finding NaN: `np.isnan`  ▶ RUN
`np.isnan(array)` gives a mask that is `True` for NaN pixels. With `~` ("not") you get the valid pixels.

In [ ]:
print("no-data pixels:", np.isnan(band).sum())
print("valid pixels:  ", (~np.isnan(band)).sum())

### 5.6 Whole numbers cannot hold NaN  ▶ RUN
NaN is a decimal (float) value. If a band is stored as whole numbers (`int`), it cannot contain NaN. The next cell makes an error **on purpose**.

In [ ]:
# This cell makes an error on purpose
counts = np.array([120, 0, 340])     # whole numbers: dtype int
counts[counts == 0] = np.nan

### 5.7 Converting to float first  ▶ RUN
`.astype(float)` converts the array to decimal numbers; then NaN is allowed.
(Some Sentinel-2 products store reflectance × 10,000 as whole numbers; they must be converted and divided by 10,000. The course files are already decimal reflectance.)

In [ ]:
counts = counts.astype(float)
counts[counts == 0] = np.nan
counts

---
## Part 6 · A real Sentinel-2 image

### 6.1 Where are the files?  ▶ RUN
As in T0, Part 5: the data folder is one level up from the tutorial folder. This is the red band (B04) of 3 September 2026:

In [ ]:
from pathlib import Path
DATA = Path("../data")
(DATA / "2026-09-03-00_00_2026-09-03-23_59_Sentinel-2_L2A_B04_(Raw).tiff").exists()

If this shows `False`, go through the path checklist of T0, Part 6.

### 6.2 Is rasterio installed?  ▶ RUN
The images are **GeoTIFF** files. They are read with the library **rasterio**.

In [ ]:
import rasterio

If you get `ModuleNotFoundError: No module named 'rasterio'`:
1. Check that Jupyter was started from the `ml4eo` environment (T0, Part 1). This is the usual cause.
2. If it really is missing: close Jupyter, open the **Anaconda Prompt** and type `conda activate ml4eo`, then `conda install -c conda-forge rasterio`. On Windows, use `conda`, not `pip`.

### 6.3 A ready-made reading function  ▶ RUN
Run this cell; you do not need to understand it yet. It defines a function (T1, Part 8) that reads one band of one date and returns it as a NumPy array.
Tutorial **T3** explains every line; the practicals use the same idea in their function `load_band()`.

In [ ]:
def read_band(date, band):
    # Read one Sentinel-2 band of one date from ml4eo/data as a NumPy array (explained in Tutorial T3)
    path = DATA / f"{date}-00_00_{date}-23_59_Sentinel-2_L2A_{band}_(Raw).tiff"
    with rasterio.open(path) as src:
        return src.read(1)

### 6.4 The whole red band  ▶ RUN

In [ ]:
full_red = read_band("2026-09-03", "B04")
print(full_red.shape, full_red.dtype)
print("number of pixels:", full_red.size)

1,702 rows × 2,380 columns: about 4 million pixels, stored as decimal numbers (`float32`). Everything from Parts 1–5 works on this array.

### 6.5 Our study area  ▶ RUN
To keep the calculations quick, we work on a window of 400 × 400 pixels in the south-east of the image: a lake with wetlands. The red frame shows where it is.
(`plt.imshow` shows an array as an image; `plt.plot` draws the frame. Plotting is Tutorial T5.)

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(full_red, cmap="gray", vmin=0, vmax=0.15)
plt.plot([1980, 2380, 2380, 1980, 1980], [1302, 1302, 1702, 1702, 1302], color="red")
plt.show()

### 6.6 Cutting out the window: slicing  ▶ RUN
Exactly as in Part 2, 2.6: rows 1302 to 1701 and columns 1980 to 2379. Do the same for the near-infrared band (B08).

In [ ]:
red = full_red[1302:1702, 1980:2380]
nir = read_band("2026-09-03", "B08")[1302:1702, 1980:2380]
print(red.shape, nir.shape)

400 rows × 400 columns. Each pixel is about 20 m × 20 m, so the window covers about 400 × 20 m = **8 km** on each side.

### 6.7 One pixel and one window  ▶ RUN
Exactly as with the tiny image of Part 2.

In [ ]:
print("pixel (100, 100):", red[100, 100])
print("window shape:", red[100:110, 100:110].shape)

### 6.8 No-data pixels  ▶ RUN
How many pixels have the value 0 (no data)? Then replace them by NaN.

In [ ]:
print("pixels with value 0:", (red == 0).sum())
red[red == 0] = np.nan
nir[nir == 0] = np.nan

### 6.9 A closer look  ▶ RUN
The no-data pixels (NaN) are left white, along the right and bottom edges.

In [ ]:
plt.imshow(red, cmap="gray", vmin=0, vmax=0.15)
plt.colorbar(label="red reflectance (B04)")
plt.show()

The dark areas are water: a lake with arms running east–west, and wetlands along its shores.

### 6.10 NDVI of 160,000 pixels in one line  ▶ RUN
The same line as in Part 1, now on two 400 × 400 arrays.

In [ ]:
ndvi = (nir - red) / (nir + red)
print(ndvi.shape)

### 6.11 NDVI map  ▶ RUN

In [ ]:
plt.imshow(ndvi, cmap="RdYlGn", vmin=-0.5, vmax=0.9)
plt.colorbar(label="NDVI")
plt.show()

### 6.12 NDVI statistics  ▶ RUN
Remember: `nan` functions, because of the no-data pixels.

In [ ]:
print("mean NDVI:  ", round(np.nanmean(ndvi), 3))
print("median NDVI:", round(np.nanmedian(ndvi), 3))
print("highest:    ", round(np.nanmax(ndvi), 3))

### 6.13 Shares of the area  ▶ RUN
The share of pixels with NDVI > 0.5 among the **valid** pixels. NaN compared with a number gives `False`, so we divide by the number of valid pixels, not by the number of all pixels.

In [ ]:
valid = ~np.isnan(ndvi)
dense_share = (ndvi > 0.5).sum() / valid.sum()
print(f"dense vegetation: {dense_share:.1%} of the valid pixels")

`:.1%` in an f-string shows a fraction as a percentage with one decimal.

### 6.14 From pixels to hectares  ▶ RUN
One pixel is about 20 m × 20 m = 400 m² = 0.04 ha (T3 uses the exact pixel size).

In [ ]:
pixel_area_ha = 20 * 20 / 10_000
dense_ha = (ndvi > 0.5).sum() * pixel_area_ha
print(f"dense vegetation: {dense_ha:.0f} ha")

(`10_000` is the same as `10000`; the `_` only makes it easier to read.)

### 6.15 Water with the MNDWI  ✍ EXERCISE
1. Read bands **B03** (green) and **B11** (SWIR) for the same window (as in 6.6), and replace 0 by NaN in both.
2. Compute the MNDWI image and store it in `mndwi`.
3. Compute the **share of valid pixels that are water** (MNDWI > 0) and store it in `water_share`.

Some lines are given.

In [ ]:
green = read_band("2026-09-03", "B03")[1302:1702, 1980:2380]
swir = read_band("2026-09-03", "B11")[1302:1702, 1980:2380]
# ✍ replace 0 by NaN in green and swir (two lines, as in 6.8)
green[green == 0] = np.nan
swir[swir == 0] = np.nan

mndwi = (green - swir) / (green + swir)          # ✍ replace None
water_share = (mndwi > 0).sum() / (~np.isnan(mndwi)).sum()    # ✍ replace None


Check your result  ▶ RUN

In [ ]:
ok_nan = np.isnan(green).sum() > 0 and np.isnan(swir).sum() > 0
check(ok_nan, "green[green == 0] = np.nan, and the same for swir")
check(mndwi is not None and np.isnan(mndwi).sum() == np.isnan(red).sum(), "mndwi = (green - swir) / (green + swir), after replacing 0 by NaN")
check(water_share is not None and 0.15 < water_share < 0.30, "(mndwi > 0).sum() divided by the number of valid pixels, (~np.isnan(mndwi)).sum()")
print(f"water: {water_share:.1%} of the valid pixels" if water_share is not None else "")

### 6.16 A simple classified map  ▶ RUN
Combine the two indices with `np.where`: water (MNDWI > 0) = 0, dense vegetation (NDVI > 0.5) = 2, all the rest = 1.

In [ ]:
mndwi = (green - swir) / (green + swir)     # computed again, in case 6.15 is not finished
lc = np.where(mndwi > 0, 0, np.where(ndvi > 0.5, 2, 1))
plt.imshow(lc, cmap="viridis")
plt.colorbar(ticks=[0, 1, 2], label="0 water · 1 other · 2 dense vegetation")
plt.show()

Note: `np.where` puts class 1 also on the NaN pixels (NaN > 0 is `False`). In a real map you would set them back to NaN, e.g. `lc = np.where(np.isnan(ndvi), np.nan, lc)`.
This rule-based map is the starting point of the course: the practicals replace the hand-made thresholds by thresholds and rules **learned** from training data.

---
## Part 7 · Stacking bands and building the feature matrix

### 7.1 Why?
A machine-learning model in scikit-learn expects a **table**: one **row per pixel** (observation) and one **column per band** (feature).
An image is the other way round: one 2-D array per band. This part turns the image into that table, step by step.

### 7.2 Stacking bands into a 3-D array  ▶ RUN
`np.stack` puts several 2-D arrays of the same shape on top of each other. The shape becomes **(bands, rows, columns)**.

In [ ]:
blue = read_band("2026-09-03", "B02")[1302:1702, 1980:2380]
blue[blue == 0] = np.nan
stack = np.stack([blue, green, red, nir, swir])
stack.shape

### 7.3 One pixel, all bands  ▶ RUN
`stack[:, 100, 100]`: all bands (`:`), row 100, column 100. This is the **spectral signature** of one pixel.

In [ ]:
stack[:, 100, 100]

### 7.4 A true-colour image  ▶ RUN
For a picture the bands must be the **last** dimension: (rows, columns, 3) with red, green, blue in that order. `np.dstack` ("depth stack") does this.
Reflectance is low (0–0.2), so it is multiplied by 3.5 to brighten, and `np.clip` keeps the values between 0 and 1.

In [ ]:
rgb = np.dstack([red, green, blue])
rgb = np.clip(rgb * 3.5, 0, 1)
plt.imshow(rgb)
plt.show()

✎ **CHANGE:** try 2.5 or 5 instead of 3.5.

### 7.5 From image to table: `reshape`  ▶ RUN
`.reshape(5, -1)` keeps the 5 bands and flattens rows × columns into one long dimension (`-1` means "work out this number yourself": 400 × 400 = 160,000).

In [ ]:
flat = stack.reshape(5, -1)
flat.shape

### 7.6 Transpose: pixels as rows  ▶ RUN
`.T` (transpose) swaps rows and columns, so that each **row is a pixel** and each **column a band**. In machine learning this table is called the **feature matrix X**.

In [ ]:
X = flat.T
X.shape

### 7.7 Removing pixels with no data  ▶ RUN
scikit-learn does not accept NaN. `np.isnan(X).any(axis=1)` is `True` for every row (pixel) with at least one NaN; `~` keeps the others.

In [ ]:
has_nan = np.isnan(X).any(axis=1)
X_valid = X[~has_nan]
print("all pixels:  ", X.shape)
print("valid pixels:", X_valid.shape)

### 7.8 Back from table to image
After a model has predicted one value per valid pixel, the predictions are put back into a 400 × 400 image with `.reshape(400, 400)`. The practicals do this inside helper functions; you only need to recognise the idea.

### 7.9 Shapes  ✍ EXERCISE
Answer with numbers (no code needed, but you may use code to find out):
1. `shape_stack`: the shape of `np.stack([red, nir])` (a tuple, e.g. `(1, 2, 3)`);
2. `n_columns`: the number of columns of the feature matrix built from these two bands.

In [ ]:
shape_stack = (2, 400, 400)     # ✍ replace None
n_columns = 2       # ✍ replace None

check(shape_stack == (2, 400, 400), "(bands, rows, columns) = (2, 400, 400)")
check(n_columns == 2, "one column per band (feature)")


---
## Part 8 · Common errors

### 8.1 Arrays of different shapes  ▶ RUN
Pixel-by-pixel arithmetic needs arrays of the **same shape**. The next cell makes an error **on purpose**: a 400 × 400 band and a 10 × 10 window.

In [ ]:
# This cell makes an error on purpose
nir - red[0:10, 0:10]

`operands could not be broadcast together with shapes (400,400) (10,10)`: the shapes do not fit. Check `.shape` of both arrays. In the practicals this happens when bands of **different resolution** (10 m and 20 m) are combined.

### 8.2 Index out of bounds  ▶ RUN
The next cell makes an error **on purpose**: there is no row 400 (rows are 0 to 399).

In [ ]:
# This cell makes an error on purpose
red[400, 0]

### 8.3 A second name is not a copy  ▶ RUN
`b = a` does **not** copy an array; it gives the same array a second name. Changing `b` changes `a` too.

In [ ]:
a = np.array([0.1, 0.0, 0.3])
b = a                 # a second name for the same array
b[b == 0] = np.nan
print("a:", a)        # a has changed too!

### 8.4 `.copy()`  ▶ RUN
Use `.copy()` when you want to change an array but keep the original.

In [ ]:
a = np.array([0.1, 0.0, 0.3])
b = a.copy()          # an independent copy
b[b == 0] = np.nan
print("a:", a)        # a is unchanged

### 8.5 Warnings are not errors
`RuntimeWarning: invalid value encountered in divide` (pink box) appears when, for example, 0 / 0 occurs in an NDVI. The cell has still run; the result contains NaN at those pixels. Replace no-data values by NaN **before** computing indices and the warning disappears.

| Error or warning | Typical cause | Fix |
|---|---|---|
| `operands could not be broadcast together` | arrays of different shape | compare `.shape`; cut or resample to the same grid |
| `IndexError: index 400 is out of bounds` | row/column number too high | check `.shape`; counting starts at 0 |
| `ValueError: cannot convert float NaN to integer` | NaN put into a whole-number array | `.astype(float)` first |
| `ValueError: The truth value of an array … is ambiguous` | `and`/`or` with arrays | `(cond1) & (cond2)`, `(cond1) \| (cond2)` |
| result is `nan` | NaN in the data | `np.nanmean`, `np.nanmax`, … |
| original array changed unexpectedly | `b = a` instead of a copy | `b = a.copy()` |
| `RuntimeWarning: invalid value encountered in divide` | 0 / 0 | set no-data to NaN first |

---
## Final challenge  ✍ EXERCISE

The lake is surrounded by wetland vegetation. In September, at the end of the dry season, it is the vegetation that is still green **and** moist.
The moisture of vegetation is measured with the **NDMI** (normalised difference moisture index) = (NIR − SWIR) / (NIR + SWIR), using B08 and B11.

1. Compute the NDMI image `ndmi` from `nir` and `swir`.
2. Make the mask `wetland`: dense vegetation (NDVI > 0.5) **and** moist (NDMI > 0.2). Use `&`, with brackets around each condition.
3. Compute its area in hectares and store it in `wetland_ha`.
4. Show the mask with `plt.imshow`. Where are the wetland pixels?

In [ ]:
ndmi = (nir - swir) / (nir + swir)             # ✍ replace None
wetland = (ndvi > 0.5) & (ndmi > 0.2)          # ✍ replace None
wetland_ha = wetland.sum() * 0.04       # ✍ replace None
# ✍ show the mask with plt.imshow (see 6.9), then plt.show()
plt.figure(figsize=(5, 5))
plt.imshow(wetland, cmap="Blues")
plt.title(f"Wetland vegetation: {wetland_ha:.1f} ha")
plt.axis('off')
plt.show()

check(wetland is not None and wetland.sum() == ((ndvi > 0.5) & ((nir - swir) / (nir + swir) > 0.2)).sum(), "ndmi = (nir - swir) / (nir + swir); wetland = (ndvi > 0.5) & (ndmi > 0.2)")
check(wetland_ha is not None and abs(wetland_ha - ((ndvi > 0.5) & ((nir - swir) / (nir + swir) > 0.2)).sum() * 0.04) < 1e-6, "number of pixels × 0.04 ha")


Expected: about 180 ha, mostly in the wetlands along the lake shores (centre and south-east of the subset). Small patches on the hills are moist trees or crops; a rule this simple cannot tell them apart. A trained classifier (Lectures 6b and 6c) uses all bands at once.

---
## Summary

| Concept | Example |
|---|---|
| array | `np.array([0.06, 0.10])`; arithmetic element by element |
| shape | `band.shape` → `(rows, columns)`; `stack.shape` → `(bands, rows, columns)` |
| indexing, slicing | `band[row, col]`, `band[0:100, 200:300]`, `band[:, 5]` |
| statistics | `.mean()`, `.min()`, `.max()`, `np.median()`, `np.percentile()`; `axis=0/1` |
| mask | `ndvi > 0.5`; `.sum()` counts, `.mean()` gives the share; `band[mask]` |
| several conditions | `(a > 0) & (b < 1)`, `\|` for "or" |
| classify | `np.where(condition, value_if_true, value_if_false)` |
| no data | `band[band == 0] = np.nan`; `np.nanmean`; `np.isnan` |
| bands → table | `np.stack([...])` → `.reshape(n_bands, -1).T` → remove NaN rows |
| copy | `b = a.copy()` |

**Next:** Tutorial **T3 · Reading satellite images with rasterio**: opening the full GeoTIFF files of the course, their metadata (resolution, coordinate reference system), and the `load_band()` and `read_scl()` functions of the practicals.